In [1]:
"""
Se si dovesse bloccare fare
>> Shut down all kernels
>> poi lanciare il seguente codice (in cella o direttamente bash)
%%bash
pkill -9 -f ollama
pkill -9 -f qdrant

"""


import sys
import subprocess
import time
import json
import urllib.request
from pathlib import Path

# Garantisce che la directory radice sia presente nel sys.path
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

from langchain_ollama import ChatOllama
# Importa i parametri centralizzati dal config.py
from src.config import LLM_MODEL, DEFAULT_KEEP_ALIVE, DEFAULT_NUM_THREAD

def start_backend_services(models_to_warm=None, keep_alive=DEFAULT_KEEP_ALIVE, num_thread=DEFAULT_NUM_THREAD):
    """
    Avvia Qdrant e Ollama, libera la RAM dai modelli inattivi e pre-carica 
    esclusivamente il modello specificato impostando i thread CPU.
    """
    if models_to_warm is None:
        models_to_warm = [LLM_MODEL]

    # ------------------------------------------------------------------
    # 1. VERIFICA E AVVIO QDRANT
    # ------------------------------------------------------------------
    qdrant_check = subprocess.run(["pgrep", "-f", "qdrant"], capture_output=True)
    if not qdrant_check.stdout:
        print(">> Avvio di Qdrant in corso...")
        subprocess.Popen(
            "cd ~/tesi_graphrag/data && nohup ~/.local/bin/qdrant > ~/tesi_graphrag/logs/qdrant.log 2>&1 &",
            shell=True
        )
    else:
        print("!>> Qdrant è già attivo.")

    # ------------------------------------------------------------------
    # 2. VERIFICA E AVVIO OLLAMA DAEMON
    # ------------------------------------------------------------------
    ollama_check = subprocess.run(["pgrep", "-f", "ollama serve"], capture_output=True)
    if not ollama_check.stdout:
        print(">> Avvio del daemon Ollama in corso...")
        subprocess.Popen(
            "nohup ~/.local/bin/ollama serve > ~/tesi_graphrag/logs/ollama.log 2>&1 &",
            shell=True
        )
    else:
        print("!>> Ollama daemon è già attivo.")

    # ------------------------------------------------------------------
    # 3. ATTESA RESPONSIVITÀ OLLAMA HTTP
    # ------------------------------------------------------------------
    print(">> Verifico connessione HTTP a Ollama...")
    server_ready = False
    for _ in range(10):
        try:
            with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as resp:
                if resp.status == 200:
                    server_ready = True
                    break
        except Exception:
            time.sleep(1)

    # ------------------------------------------------------------------
    # 3.5 PULIZIA RAM: Scarica i modelli attivi diversi da quello scelto
    # ------------------------------------------------------------------
    if server_ready:
        try:
            with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
                ps_data = json.loads(resp.read().decode("utf-8"))
                for running in ps_data.get("models", []):
                    model_name = running.get("name")
                    # Se il modello in RAM non è tra quelli che vogliamo scaldare, scaricalo
                    if model_name not in models_to_warm and f"{model_name}:latest" not in models_to_warm:
                        print(f">> Scarico dalla RAM il modello inattivo '{model_name}'...")
                        unload_payload = json.dumps({"model": model_name, "keep_alive": 0}).encode("utf-8")
                        unload_req = urllib.request.Request(
                            "http://localhost:11434/api/generate",
                            data=unload_payload,
                            headers={"Content-Type": "application/json"}
                        )
                        urllib.request.urlopen(unload_req, timeout=10)
        except Exception as e:
            print(f"!>> Nota sulla pulizia RAM: {e}")

    # ------------------------------------------------------------------
    # 4. PRE-CARICAMENTO MODELLI IN RAM (con Keep-Alive e num_thread)
    # ------------------------------------------------------------------
    if server_ready:
        for model in models_to_warm:
            print(f">> Pre-caricamento in RAM di '{model}' (keep_alive='{keep_alive}', num_thread={num_thread})...")
            
            payload = json.dumps({
                "model": model, 
                "keep_alive": keep_alive,
                "options": {
                    "num_thread": num_thread
                }
            }).encode("utf-8")
            
            req = urllib.request.Request(
                "http://localhost:11434/api/generate",
                data=payload,
                headers={"Content-Type": "application/json"}
            )
            
            try:
                with urllib.request.urlopen(req, timeout=180) as resp:
                    print(f"!>> Modello '{model}' pronto in RAM con {num_thread} thread.")
            except Exception as e:
                print(f"!!!>> Errore durante il caricamento di '{model}': {e}")
    else:
        print("!!!>>> Impossibile raggiungere il server Ollama sulla porta 11434.")

    # ------------------------------------------------------------------
    # 5. REPORT FINALE DELLO STATO
    # ------------------------------------------------------------------
    print("\n--- Processi di Sistema ---")
    subprocess.run("ps aux | grep -E 'qdrant|ollama' | grep -v grep", shell=True)

    print("\n--- Modelli Attivi in RAM (ollama ps) ---")
    subprocess.run("~/.local/bin/ollama ps", shell=True)


def get_llm(model=LLM_MODEL, keep_alive=DEFAULT_KEEP_ALIVE, num_thread=DEFAULT_NUM_THREAD):
    """
    Funzione helper per istanziare ChatOllama ottimizzato per CPU.
    """
    return ChatOllama(
        model=model,
        keep_alive=keep_alive,
        num_thread=num_thread
    )


# ======================================================================
# SELEZIONA QUI COSA AVVIARE
# ======================================================================

# Predefinito: Usa il modello selezionato centralmente in src/config.py
SELECTED_MODEL = LLM_MODEL

# Per sovrascrivere direttamente qui senza toccare config.py,
# scommenta una delle righe seguenti:
# SELECTED_MODEL = "llama3.2"       # OPZIONE A: Llama 3.2 (3B) -> Sviluppo fluido e bilanciato
# SELECTED_MODEL = "llama3.2:1b"    # OPZIONE B: Llama 3.2 (1B) -> Ultra-veloce per debug rapido
# SELECTED_MODEL = "llama3.1"       # OPZIONE C: Llama 3.1 (8B) -> Massima qualità per test finali


# ======================================================================
# ESECUZIONE ED INIZIALIZZAZIONE AUTOMATICA
# ======================================================================

# 1. Avvia i servizi, pulisce la RAM ed invia in RAM solo il modello selezionato
start_backend_services(models_to_warm=[SELECTED_MODEL], keep_alive=DEFAULT_KEEP_ALIVE, num_thread=DEFAULT_NUM_THREAD)

# 2. Istanzia l'oggetto LLM pronto all'uso per LangChain
llm = get_llm(model=SELECTED_MODEL, keep_alive=DEFAULT_KEEP_ALIVE, num_thread=DEFAULT_NUM_THREAD)

# 3. Test di verifica dello streaming veloce
print(f"\n>> Test velocità di generazione ({SELECTED_MODEL}):")
for chunk in llm.stream("Rispondi OK se sei pronto"):
    print(chunk.content, end="", flush=True)
print()

>> Avvio di Qdrant in corso...
>> Avvio del daemon Ollama in corso...
>> Verifico connessione HTTP a Ollama...
>> Pre-caricamento in RAM di 'llama3.2' (keep_alive='2h', num_thread=4)...
!>> Modello 'llama3.2' pronto in RAM con 4 thread.

--- Processi di Sistema ---
jovyan    758676  8.0  0.1 4118128 610256 ?      Sl   11:40   0:00 /home/jovyan/.local/bin/qdrant
jovyan    758679 10.4  0.0 2404300 56756 ?       Sl   11:40   0:00 /home/jovyan/.local/bin/ollama serve
jovyan    758775  139  0.4 4651144 2514388 ?     Sl   11:41   0:05 /home/jovyan/.local/lib/ollama/llama-server --model /home/jovyan/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff --port 41093 --host 127.0.0.1 --no-webui --offline -c 4096 -np 1 --log-verbosity 4 --no-log-prefix --no-log-timestamps --no-jinja --chat-template chatml --load-mode none --flash-attn auto -b 512 -ub 512 -t 4 --context-shift --keep 4

--- Modelli Attivi in RAM (ollama ps) ---
NAME               ID          